In [1]:
# import cython
import numpy as np
import matplotlib.pyplot as plt 
import glob
from ISRpy import beampack
jro0 = beampack.jrobeam

from skimage.morphology import skeletonize
import torch

In [2]:
from OXmodeCython import OXmode_cython

In [3]:
valleyfolder = "/rd0/home/linle2/16_Valley_spectra/valley7_redo/spectra_d2016014/proc2/"
datafiles = glob.glob1(valleyfolder+'/','*.npy') 
datafiles.sort()

valleytime = []
valleyname = []
for filename in datafiles:
    fullname = valleyfolder+'/'+filename
    data=np.load(fullname,allow_pickle=True,fix_imports=True)[()]
    filetime=data['timearray']
    lt = filetime.tm_hour+filetime.tm_min/60.0+filetime.tm_sec/3600.0
    valleytime += [lt]
    valleyname += [filename]
valleytime = np.array(valleytime)
valleyname = np.array(valleyname)

In [4]:
import datetime
vipirfolder = '/home/jackle/Documents/MLvipir/VIPIR_PROC/org/'
datafiles = glob.glob1(vipirfolder+'/','*.npy') 
datafiles.sort()
vipir_2_valley_dict = {}
for filename in datafiles:
    vipirTimeSct = np.load(vipirfolder+filename,allow_pickle=True)[()]['vipirTimeSct']
    dt0 = datetime.datetime(vipirTimeSct['start_year'][0],vipirTimeSct['start_month'][0],vipirTimeSct['start_day'][0])

    vipirLT=vipirTimeSct['start_hour'][0]-5.0+vipirTimeSct['start_minute'][0]/60.0+vipirTimeSct['start_second'][0]*1.0/3600.0
    valley_idx = np.argmin(np.abs(valleytime-vipirLT))
    if (valleytime[valley_idx]-vipirLT) < 1/60:
        valleyfile=valleyname[valley_idx] # select valley file matching VIPIRgram
        valleyLT = valleytime[valley_idx]
        vipir_2_valley_dict[filename] = valleyfile

igrf_year = dt0.year + (dt0-datetime.datetime(dt0.year,1,1)).total_seconds() / (
            datetime.datetime(dt0.year+1,1,1)-datetime.datetime(dt0.year,1,1)).total_seconds()

In [5]:
len(vipir_2_valley_dict)

1747

In [6]:
npyfilename = 'JM91J_2016014141141.npy'
result_npy = np.load(npyfilename, allow_pickle=True, fix_imports=True)[()]

ionogram = result_npy['mask']
freqs = result_npy['vipir_freqs']
hts = result_npy['vipir_hts']

# plt.imshow(ionogram, origin='lower')

phasefolder = '/home/jackle/Documents/ValleyTest/phase_plots/'

In [7]:
valleytx = 0.037109375
valleyty = -0.037109375

v_fes = []
v_theta = []

npyfilename = 'JM91J_2016014141141.npy'
result_npy = np.load(valleyfolder + vipir_2_valley_dict[npyfilename], allow_pickle=True, fix_imports=True)[()]
valleyz = result_npy['hts']

for zz in valleyz/1000:
    r,lon,lat,dec,ha,aspect,B = jro0.aspect_txty(igrf_year,zz,valleytx,valleyty)
    v_fes += [(np.dot(B,B))**0.5*1e-9*1.602e-19/9.1093e-31]
    v_theta += [aspect]

valley_fes = np.array(v_fes,dtype=np.float64) /1e6/2/np.pi
valley_theta = np.array(v_theta) 

def phase_sim(valley_fps: torch.Tensor, valley_fes: torch.Tensor, valley_theta: torch.Tensor, valleyz: torch.Tensor): 
    valleyNe = valley_fps**2/8.06 * 1e11
    Phase_sim = 2.0*80.6/(3e8*5e7**2)*valley_fes[1:]*2*np.pi*1e6*torch.abs(torch.cos(valley_theta[1:]))*valleyNe[1:]*(valleyz[1:]-valleyz[:-1])*1e3
    Phase_sim = torch.cumsum(Phase_sim,dim=0)
    return Phase_sim

def phase_loss(Valley_phase, Valley_weight, Phase_sim):
    offset2 = torch.mean(Phase_sim[170:])
    phase_error = torch.zeros(4)
    for idx in range(len(Valley_phase)):
        phase = torch.from_numpy(Valley_phase[idx])
        weight = np.zeros(Valley_phase[idx].shape)
        weight[Valley_weight[idx]] = 1
        weight = torch.from_numpy(weight)
        offset1 = torch.mean(phase[171:])
        phase_error[idx] += torch.sum(torch.abs(phase[171:] - offset1 - (Phase_sim[170:] - offset2))**2 * weight[171:])

    return torch.min(phase_error)

In [88]:
def softplus(x,beta=7):
        return 1/beta*torch.log(1+torch.exp(x*beta))

def fps_parabolic(a,z):
    para1 = a[0]-((z-a[1])/a[2]/3)**2
    # para2 = a[3]-((z-a[4])/a[5]/3)**2 + softplus(para2)
    # para3 = a[6]-((z-a[7])/a[8]/3)**2 + softplus(para3)

    fps1 = (80.6*(softplus(para1, beta=a[3]) ))**0.5
    # fps2 = (80.6*np.max(np.vstack([para1,para2,para3]), axis=0))**0.5
    return fps1#,fps2

def fps_gg(a,z):
    para1 = a[0] * torch.exp(-((z-a[1])/a[2]) ** 2)
    # para2 = a[3]-((z-a[4])/a[5]/3)**2 + softplus(para2)
    # para3 = a[6]-((z-a[7])/a[8]/3)**2 + softplus(para3)

    fps1 = (80.6*para1)**0.5
    # fps2 = (80.6*np.max(np.vstack([para1,para2,para3]), axis=0))**0.5
    return fps1#,fps2

def evalutate_phase(x_vals, Valley_phase, Valley_weight ,fps_func, valleyz):
    valley_fps = fps_func(x_vals)
    p_sim = phase_sim(valley_fps, torch.from_numpy(valley_fes), torch.from_numpy(valley_theta), torch.from_numpy(valleyz))
    p_loss = phase_loss(Valley_phase, Valley_weight, p_sim)
    # print(p_loss)
    return p_loss


In [89]:
from torch.optim import Adam
import scipy.constants as const
from matplotlib import rcParams
import datetime

In [97]:
dz=0.5 # resolution in km --- 0.1 is for good resolution, 0.5 for speed

#get a list of points to fit a spline
N = 12
zmin = 0
zmax = 450
Z = torch.arange(0.01,zmax,dz)

fes = []
for iZ in Z:
    r,geoc_lat,lon,aspect,B = jro0.aspect_angle(igrf_year,jro0.xyz0 + jro0.zenith0 * iZ.numpy())
    fes += [const.e * np.linalg.norm(B) * 1e-9 / const.m_e /1e6 /2/np.pi] #MHz
fes = np.array(fes)

def grad_descent_on_seg(filename):
    outdir = 'anchor_test'
# for filename in filenames:
    # processed_files = glob.glob1('testset_fitting_transformer/','*.npy')
    # if filename in processed_files:
    #     return

    if filename in vipir_2_valley_dict:
        data = np.load(phasefolder + vipir_2_valley_dict[filename], allow_pickle=True, fix_imports=True)[()]
        Valley_phase = data['phase_unwrapped']
        Valley_goodidx = data['phase_good_idx']
    else:
        return 

    data = np.load('seg_result_transformer/'+filename,allow_pickle=True)[()]
    img = data['image']
    imgO = skeletonize(data['imgO'])
    imgX = skeletonize(data['imgX'])

    Y, X = imgO.nonzero()
    fvsO0 = freqs[X]
    vhsO0 = hts[Y]
    targetO = np.vstack((fvsO0,vhsO0)).T

    Y, X = imgX.nonzero()
    fvsX0 = freqs[X]
    vhsX0 = hts[Y]
    targetX = np.vstack((fvsX0,vhsX0)).T

    fof2 = fvsO0.max()

    lossfucntion_phase = lambda xv: evalutate_phase(xv, Valley_phase, Valley_goodidx, lambda xv: fps_gg(xv, torch.from_numpy(valleyz)), valleyz)
    # xvals = torch.tensor([0.9,400,60,0.2,300,70,0.04,150,20], requires_grad=True)
    # xvals = torch.tensor([0.9,400,60,0.04,150,20], requires_grad=True)
    xvals = torch.tensor([0.9,400,60], requires_grad=True)
    # fps = fps_gg(xvals, Z)
    # plt.plot(fps.detach())
    
    losses = []
    all_xvals = []
    lr = 1e-2
    grad_limit = 1e3
    grad_mem = 0.1
    wt_decay = 5e-4

    optimizer = Adam([xvals], lr=1e-3)

    for it in range(50000):
        optimizer.zero_grad()
        loss = lossfucntion_phase(xvals) # 
        loss.backward()
        # print(xvals.grad)
        optimizer.step()
        losses += [loss.item()]
        all_xvals += [xvals.detach().numpy()]

    fps = fps_gg(xvals, Z)
    p_sim = phase_sim(fps_gg(xvals, torch.from_numpy(valleyz)), torch.from_numpy(valley_fes), torch.from_numpy(valley_theta), torch.from_numpy(valleyz))
    # print(xvals)
    np.save(f'{outdir}/{filename}', {'filename':filename, 'image':img, 'imgO': imgO, 'imgX': imgX, 'all_xvals': all_xvals, 'all_losses':losses}) #'zvals': z,

    fig,axes = plt.subplots(1,4,figsize=(18,5),sharex=False)
    ax1 = axes[0]
    # ax1.set_ylim(0,700)
    temp = np.copy(imgO)
    temp[imgX > 0] = 2
    ax1.pcolormesh(freqs,hts,temp,shading='auto')
    m, = ax1.plot (fps.detach().numpy(), Z.numpy(), 'k-', label='spline fp')
    fvsO,vhsO,fvsX,vhsX=OXmode_cython(fps.detach().numpy().astype(np.double), fes, dz)
    mO, = ax1.plot(fvsO, vhsO, 'r-', label='O-trace hv')
    mX, = ax1.plot(fvsX, vhsX, 'b-', label='X-trace hv')

    ax2 = axes[1]
    # ax2.set_ylim(0,700)
    ax2.pcolormesh(freqs,hts,img,shading='auto')
    ax3 = axes[2]
    ax3.plot(losses) 
    
    ax4 = axes[3]
    ax4.plot(p_sim.detach().numpy(), valleyz[:-1], c='r')
    ax4.scatter(Valley_phase[2], valleyz, marker='.')
    ax4.set_ylim(0,700)
    
    fig.savefig(f'{outdir}/{filename[0:-4]}.png', bbox_inches='tight', dpi=300)
    plt.close()
    return

In [98]:
from joblib import Parallel, delayed

filenames = glob.glob1('seg_result_transformer/','*.npy')
filenames.sort()
# grad_descent_on_seg(filenames[100])
Parallel(n_jobs=16)(delayed(grad_descent_on_seg)(f) for f in filenames[:])


/home/jackle/.local/python311/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [ ]:
from matplotlib import rcParams
import scipy.interpolate as inter
import datetime
import scipy.constants as const

import time

dt0 = datetime.datetime(2015,4,15)
igrf_year = dt0.year + (dt0-datetime.datetime(dt0.year,1,1)).total_seconds() / (
        datetime.datetime(dt0.year+1,1,1)-datetime.datetime(dt0.year,1,1)).total_seconds()

dz=0.5 # resolution in km --- 0.1 is for good resolution, 0.5 for speed

#get a list of points to fit a spline
N = 12
zmin = 0
zmax = 450
z = np.linspace(zmin,zmax,N)

#spline fit
# xvals_gold=[0.,0.22222222,2.4444444,4.33333333,5.6666666,7.22222222,8.88888889,
    #    10.2222222,9.88888889,8.11111111,6.15,3.5875,0.,0.,0.,0.,0.]
xvals_gold=[0.,0.22222222,2.4444444,4.33333333,5.6666666,7.22222222,8.88888889,
       10.2222222,9.88888889,8.11111111,6.15,3.5875,]
# spline = inter.PchipInterpolator(z,xvals_gold)
spline = inter.CubicSpline(z,xvals_gold)

#figure.subplot.right
rcParams['figure.subplot.right'] = 0.78
#set up a plot
fig,axes = plt.subplots(1,1,figsize=(9.0,8.0),sharex=True)
ax1 = axes

# height grid
Z = np.arange(0.01,zmax,dz)

fes = []
for iZ in Z:
    r,geoc_lat,lon,aspect,B = jro0.aspect_angle(igrf_year,jro0.xyz0 + jro0.zenith0 * iZ)
    fes += [const.e * np.linalg.norm(B) * 1e-9 / const.m_e /1e6 /2/np.pi] #MHz
fes = np.array(fes)

fps_gold=spline(Z)
# ax1.pcolormesh(freqs,hts,log10(ionograms[3,:,:]),shading='auto')
l, = ax1.plot (xvals_gold,z,color='b',linestyle='none',marker='o',markersize=8)
m, = ax1.plot (spline(Z), Z, 'r-', label='spline fp')

fvsO0,vhsO0,fvsX0,vhsX0=OXmode_cython(fps_gold, fes, dz)
mO, = ax1.plot(fvsO0, vhsO0, 'g-', label='O-trace hv')
mX, = ax1.plot(fvsX0, vhsX0, 'y-', label='X-trace hv')

# xvals = np.array(xvals_gold) + 0.5

xvals = np.array([0.9,400,60,0.2,300,70,0.04,150,20])

fps = fps_parabolic(xvals,Z)
m, = ax1.plot (fps, Z, 'k-', label='spline fp')
fvsO,vhsO,fvsX,vhsX=OXmode_cython(fps, fes, dz)
mO, = ax1.plot(fvsO, vhsO, 'r-', label='O-trace hv')
mX, = ax1.plot(fvsX, vhsX, 'b-', label='X-trace hv')

plt.ylim(0,800) 


In [74]:
# resize_back = transforms.Resize(ionogram.shape)
# resize_tr = transforms.Resize((572, 572))

def grad_descent_on_seg(filename):
    outdir = 'anchor_test'
# for filename in filenames:
    # processed_files = glob.glob1('testset_fitting_transformer/','*.npy')
    # if filename in processed_files:
    #     return

    if filename in vipir_2_valley_dict:
        Valley_phase = np.load(phasefolder + vipir_2_valley_dict[filename], allow_pickle=True, fix_imports=True)[()]['phase_unwrapped']
    else:
        return 

    data = np.load('seg_result_transformer/'+filename,allow_pickle=True)[()]
    img = data['image']
    imgO = skeletonize(data['imgO'])
    imgX = skeletonize(data['imgX'])

    Y, X = imgO.nonzero()
    fvsO0 = freqs[X]
    vhsO0 = hts[Y]
    targetO = np.vstack((fvsO0,vhsO0)).T

    Y, X = imgX.nonzero()
    fvsX0 = freqs[X]
    vhsX0 = hts[Y]
    targetX = np.vstack((fvsX0,vhsX0)).T

    # # evalutate(xv,  targetO, targetX ,lambda xv: fps_parabolic(xv, Z)) + 
    # lossfucntion_phase = lambda xv: 5*evalutate_phase(xv, Valley_phase ,lambda xv: fps_parabolic(xv, valleyz))
    # xvals = np.array([0.9,400,60,0.2,300,70,0.04,150,20])

    # losses = []
    # all_xvals = []
    # lr = 1e-2
    # grad_limit = 1e3
    # grad_mem = 0.1
    # wt_decay = 5e-4

    # grad = 0
    # for it in range(200):
    #     # try:
    #     grad_cur, loss = differential(lossfucntion_phase , x_vals=xvals) # 
    #     grad_cur = np.clip(grad_cur, min=-grad_limit, max = grad_limit)
    #     grad = grad_cur #grad * gr_mem + grad_cur * (1-grad_cur)
        
    #     xvals += -1 * lr * grad #- wt_decay*x_vals
    #     if it % 40 == 0: lr *= 0.71
    #     losses += [loss]
    #     all_xvals += [xvals]
    #     # except:
    #     #     print("some error")
    #     #     print("some error")
    # fps = fps_parabolic(xvals, Z)
    # p_sim = phase_sim(fps_parabolic(xvals, valleyz), valley_fes, valley_theta, valleyz)
    
    # fof2 = fvsO0.max()
    x_vals = np.array([ 0., 1., 1.8888889300000002, 1.3333332699999998, 1.5555556199999998, 1.6666666700000006, 1.3333333099999987, -0.33333330999999866, -1.777777780000001, -1.9611111099999992, -2.5625000000000004, 0.0,])
    # x_vals = x_vals / x_vals.cumsum().max() * fof2

    # 5*regularization_loss(xv) + evalutate(xv,  targetO, targetX, Valley_phase ,lambda xv: fps_spline(xv, Z)) +
    lossfucntion = lambda xv:  5*evalutate_phase(xv, Valley_phase ,lambda xv: fps_spline(xv, valleyz))
    # losses = []
    # all_xvals = []
    # for shift in range(1,5):
    #     xv = np.pad(x_vals,(shift,0), mode='constant', constant_values=(0,0))[:x_vals.shape[0]]
    #     losses += [lossfucntion(xv)]
    #     all_xvals += [xv]
    # bestshift = np.argmin(losses)
    # x_vals = all_xvals[bestshift]

    losses = []
    all_xvals = []
    lr = 4e-2
    grad_limit = 1e3
    gr_mem = 0.1
    wd = 1e-4

    grad = 0
    for it in range(200):
        grad_cur, loss = differential(lossfucntion , x_vals=x_vals) # 
        grad_cur = np.clip(grad_cur, min=-grad_limit, max = grad_limit)
        grad = grad_cur #grad * gr_mem + grad_cur * (1-grad_cur)
        
        x_vals += -1 * lr * grad #- wd*x_vals
        if it % 40 == 0: lr *= 0.71
        losses += [loss]
        all_xvals += [x_vals]
    fps = fps_spline(x_vals, Z)
    p_sim = phase_sim(fps_spline(x_vals, valleyz), valley_fes, valley_theta, valleyz)

    np.save(f'{outdir}/{filename}', {'filename':filename, 'image':img, 'imgO': imgO, 'imgX': imgX, 'all_xvals': all_xvals, 'zvals': z, 'all_losses':losses})
    
    fig,axes = plt.subplots(1,4,figsize=(18,5),sharex=False)
    ax1 = axes[0]
    ax1.set_ylim(0,700)
    temp = np.copy(imgO)
    temp[imgX > 0] = 2
    ax1.pcolormesh(freqs,hts,temp,shading='auto')
    m, = ax1.plot (fps, Z, 'k-', label='spline fp')
    fvsO,vhsO,fvsX,vhsX=OXmode_cython(fps, fes, dz)
    mO, = ax1.plot(fvsO, vhsO, 'r-', label='O-trace hv')
    mX, = ax1.plot(fvsX, vhsX, 'b-', label='X-trace hv')

    ax1.set_ylim(0,700)
    # ax1.set_xlim(0, 12.5)
    ax2 = axes[1]
    ax2.set_ylim(0,700)
    # ax2.set_xlim(0, 12.5)
    ax2.pcolormesh(freqs,hts,img,shading='auto')
    ax3 = axes[2]
    # ax3.set_xlim(0, 100)
    ax3.plot(losses)

    ax4 = axes[3]
    ax4.plot(p_sim, valleyz, c='r')
    ax4.scatter(Valley_phase[2], valleyz, marker='.')
    ax4.set_ylim(0,700)
    
    fig.savefig(f'{outdir}/{filename[0:-4]}.png', bbox_inches='tight', dpi=300)
    plt.close()
    return
    

In [75]:
from joblib import Parallel, delayed
# from multiprocessing import Pool

filenames = glob.glob1('seg_result_transformer/','*.npy')
filenames.sort()

Parallel(n_jobs=16)(delayed(grad_descent_on_seg)(f) for f in filenames[:200])
# grad_descent_on_seg(filenames[100])
# processes_pool = Pool(5)
# processes_pool.map(grad_descent_on_seg, filenames)
# processes_pool.close()

NameError: name 'differential' is not defined